# boolean-mask-combine — worked example 1: Axis-aligned bounding-box containment mask

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `boolean-mask-combine`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

To test whether 2-D points fall inside an axis-aligned bounding box you AND together four per-coordinate comparisons: `x >= xmin`, `x <= xmax`, `y >= ymin`, `y <= ymax`. Each comparison produces a bool tensor of the same shape, and `&` combines them elementwise. Because `&` binds tighter than `>=`/`<=`, every comparison MUST be wrapped in parentheses or Python misparses the expression.

## Worked solution

**Goal:** given points `pts` of shape `(N, 2)` and a box `(xmin, ymin, xmax, ymax)`, return a `(N,)` bool mask, `True` where the point is inside the box (inclusive).

**Step 1 — split the coordinates.** `x = pts[:, 0]` and `y = pts[:, 1]` are each `(N,)`. Slicing column 0 and 1 keeps the per-point structure so every downstream comparison is `(N,)`.

**Step 2 — build the four predicates.** `(x >= xmin)`, `(x <= xmax)`, `(y >= ymin)`, `(y <= ymax)`. Each is an elementwise comparison of a `(N,)` tensor against a scalar, yielding a `(N,)` bool tensor. The parentheses are essential: `x >= xmin & x <= xmax` would parse as `x >= (xmin & x) <= xmax` and raise or silently misbehave.

**Step 3 — AND them together.** A point is inside only if ALL four constraints hold, so we combine with `&`. `&` is elementwise logical-and on bool tensors, producing a final `(N,)` bool mask. Conjunction is the right combinator because containment is the intersection of four half-plane constraints.

**Why it works:** the box is the intersection of four half-planes, and intersection of per-element predicates is exactly elementwise AND. The result dtype is bool because every operand is bool.

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat
np.random.seed(0)
t.manual_seed(0)

def inside_box(pts: Tensor, xmin: float, ymin: float, xmax: float, ymax: float) -> Tensor:
    x = pts[:, 0]
    y = pts[:, 1]
    return (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)

pts = t.tensor([[0.5, 0.5], [2.0, 0.5], [-1.0, 0.0], [1.0, 1.0]])
mask = inside_box(pts, 0.0, 0.0, 1.0, 1.0)
print(mask)
print(mask.dtype, mask.shape)